# Optional: camera movement and acquired time
Run this notebook independently after the main lesson. Use the small teaching data. You will compare a camera rotation around one frame with a movie that plays through the recorded frames. Keep the viewer open until exports finish. The helper `create_scene` simply repeats the layer calls explained in the main notebook.

In [ ]:
%gui qt
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data" / "zenodo-manifest.json").is_file())
DATA = ROOT / "data" / "teaching"
print(DATA)

In [ ]:
from napari_workshop.render import create_scene, export_time_series, fit_frame
import imageio.v2 as imageio
from napari_animation import Animation
from datetime import datetime
viewer, metadata = create_scene(DATA)
OUTPUT = ROOT / "outputs" / datetime.now().strftime("animation-%Y%m%d-%H%M%S-%f")
OUTPUT.mkdir(parents=True)

Use a small camera turn first. Rotation angles describe the view; they do not change the sample or track coordinates. Both keyframes below use acquired frame 10. You may also capture these states through the animation wizard in the Plugins menu.

The animation first saves native PNG frames in `camera-frames`. We then fit each complete frame into 640 × 480 pixels with black padding and encode at 10 playback fps. This preserves aspect ratio and avoids automatic video resizing. Keep these intermediate PNGs if you want to inspect individual views; they use additional disk space.

In [ ]:
viewer.dims.set_current_step(0, 10)
animation = Animation(viewer)
viewer.camera.angles = (0, 0, 90)
animation.capture_keyframe()
viewer.camera.angles = (25, 15, 110)
animation.capture_keyframe(steps=30)
animation.animate(str(OUTPUT / "camera-frames"), canvas_only=True)
with imageio.get_writer(OUTPUT / "rotation.mp4", fps=10, codec="libx264",
                         pixelformat="yuv420p", macro_block_size=16) as writer:
    for path in sorted((OUTPUT / "camera-frames").glob("*.png")):
        writer.append_data(fit_frame(imageio.imread(path)))
with imageio.get_reader(OUTPUT / "rotation.mp4") as reader:
    print("Camera movie:", reader.count_frames(), "frames;",
          reader.get_meta_data()["fps"], "fps;", reader.get_meta_data()["size"], "pixels")

Next, hold the camera still and export one screenshot per acquired frame. The helper streams frames to the encoder, then reads the movie to verify its frame count and dimensions. It creates a **new** subfolder and records export settings in `export.json`. Do not reuse an existing subfolder for another run.

In [ ]:
movie = export_time_series(viewer, OUTPUT / "acquired-time", metadata["timepoints"], fps=5)
print(movie)

**Compare:** in the camera animation, you move around one recorded volume. In the time-series movie, you watch successive recorded frames from a fixed viewpoint. If you combine camera movement and time, keep the frame label visible so viewers can tell them apart.

**Try it:** change playback fps to 10 and export into a new folder. How does the movie’s duration change? Does changing playback speed change how often the images were acquired? Save a preferred view before trying longer exports. PNG screenshots remain a useful fallback if video playback is unavailable.

In [ ]:
viewer.close()